# Experiment 007: Pair Universe Sweep

**Goal:** Test if reducing the pair universe from 100 to 30/40/50 concentrates capital on profitable blue chips.

**Setup:** All arms use `IchiV3_LS_Static_WhaleCap` with pool=2%, oi=2.5%, ratio=10%, minpos=1%, max_open_trades=10.
Only difference: `HistoricalVolumePairList.number_assets` = 30, 40, 50, or 100.

**Baselines:** Arm D from exp 003 (static $500K filter, 35 pairs).

In [ ]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
TEMPLATE = 'plotly_dark'

RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/007-pair-universe-sweep/results'
BASELINE_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/003-volume-pairlist-layers/results'

COLORS = {
    'Top 30': '#636EFA',
    'Top 40': '#AB63FA',
    'Top 50': '#00CC96',
    'Top 100': '#EF553B',
    'Arm D ($500K filter)': '#FFA15A',
}

STYLES = {
    'Top 30': dict(color='#636EFA', width=2),
    'Top 40': dict(color='#AB63FA', width=2),
    'Top 50': dict(color='#00CC96', width=3),
    'Top 100': dict(color='#EF553B', width=2, dash='dot'),
    'Arm D ($500K filter)': dict(color='#FFA15A', width=2, dash='dash'),
}

In [ ]:
def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades = pd.DataFrame(data['strategy'][strategy_name]['trades'])
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades


def calculate_metrics(trades, starting_balance=STARTING_BALANCE):
    ts = trades.sort_values('close_date').copy()
    total = ts['profit_abs'].sum()
    days = (ts['close_date'].max() - ts['open_date'].min()).days
    years = days / 365.25
    final_eq = starting_balance + total
    cagr = ((final_eq / starting_balance) ** (1/years) - 1) * 100 if years > 0 else 0
    eq = starting_balance + ts['profit_abs'].cumsum()
    dd = ((eq - eq.cummax()) / eq.cummax() * 100).min()
    daily = ts.groupby(ts['close_date'].dt.date)['profit_abs'].sum() / starting_balance
    sharpe = (daily.mean() / daily.std()) * np.sqrt(365) if daily.std() > 0 else 0
    down = daily[daily < 0]
    sortino = (daily.mean() / down.std()) * np.sqrt(365) if len(down) > 0 and down.std() > 0 else 0
    calmar = abs(cagr / dd) if dd != 0 else 0
    return {
        'Trades': len(ts), 'Pairs': ts['pair'].nunique(),
        'Profit ($)': round(total), 'CAGR (%)': round(cagr,1),
        'Max DD (%)': round(dd,1), 'Sharpe': round(sharpe,2),
        'Sortino': round(sortino,2), 'Calmar': round(calmar,2),
        'Win Rate (%)': round((ts['profit_abs']>0).mean()*100,1),
        'Avg Stake ($)': round(ts['stake_amount'].mean()),
        'Min Stake ($)': round(ts['stake_amount'].min()),
        'Dust (<$1K)': len(ts[ts['stake_amount']<1000]),
    }

In [ ]:
# Load all configs
configs = {
    'Top 30': RESULTS_DIR / 'arm_a_top30.zip',
    'Top 40': RESULTS_DIR / 'arm_b_top40.zip',
    'Top 50': RESULTS_DIR / 'arm_c_top50.zip',
    'Top 100': RESULTS_DIR / 'arm_d_top100.zip',
    'Arm D ($500K filter)': BASELINE_DIR / 'arm_d_volume_liq_whale.zip',
}

all_trades = {}
all_metrics = {}
for label, path in configs.items():
    trades = load_trades_from_zip(path)
    all_trades[label] = trades
    all_metrics[label] = calculate_metrics(trades)

df = pd.DataFrame(all_metrics).T
display_cols = ['Trades', 'Pairs', 'Profit ($)', 'CAGR (%)', 'Max DD (%)', 'Sharpe', 'Sortino',
                'Calmar', 'Win Rate (%)', 'Avg Stake ($)', 'Min Stake ($)', 'Dust (<$1K)']
df[display_cols].sort_values('Sharpe', ascending=False)

In [ ]:
# Equity curves
fig = go.Figure()
for label, trades in all_trades.items():
    ts = trades.sort_values('close_date').copy()
    ts['equity'] = STARTING_BALANCE + ts['profit_abs'].cumsum()
    fig.add_trace(go.Scatter(x=ts['close_date'], y=ts['equity'], mode='lines',
                             name=label, line=STYLES[label]))

fig.update_layout(title='Equity Curves — Pair Universe Comparison',
                  xaxis_title='Date', yaxis_title='Equity ($)',
                  template=TEMPLATE, height=600,
                  legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01))
fig.show()

In [ ]:
# Drawdown curves
fig = go.Figure()
for label, trades in all_trades.items():
    ts = trades.sort_values('close_date').copy()
    ts['equity'] = STARTING_BALANCE + ts['profit_abs'].cumsum()
    ts['drawdown'] = (ts['equity'] - ts['equity'].cummax()) / ts['equity'].cummax() * 100
    style = {k: v for k, v in STYLES[label].items() if k != 'width'}
    style['width'] = 1.5
    fig.add_trace(go.Scatter(x=ts['close_date'], y=ts['drawdown'], mode='lines',
                             name=label, line=style))

fig.update_layout(title='Drawdown Curves',
                  xaxis_title='Date', yaxis_title='Drawdown (%)',
                  template=TEMPLATE, height=500,
                  legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01))
fig.show()

In [ ]:
# Pair-level profit breakdown: which pairs exist in top-100 but not top-30/50?
top100_trades = all_trades['Top 100']
top50_trades = all_trades['Top 50']
top30_trades = all_trades['Top 30']

t100_pairs = set(top100_trades['pair'].unique())
t50_pairs = set(top50_trades['pair'].unique())
t30_pairs = set(top30_trades['pair'].unique())

# Profit by pair for top-100
pair_pnl = top100_trades.groupby('pair')['profit_abs'].sum().sort_values(ascending=False)

# Mark which tiers each pair belongs to
tier_data = []
for pair in pair_pnl.index:
    pnl = pair_pnl[pair]
    in_30 = pair in t30_pairs
    in_50 = pair in t50_pairs
    tier = 'Top 30' if in_30 else ('Top 31-50' if in_50 else 'Top 51-100')
    tier_data.append({'Pair': pair, 'PnL': pnl, 'Tier': tier,
                      'Trades': len(top100_trades[top100_trades['pair']==pair])})

tier_df = pd.DataFrame(tier_data)

for tier in ['Top 30', 'Top 31-50', 'Top 51-100']:
    subset = tier_df[tier_df['Tier']==tier]
    print(f'{tier}: {len(subset)} pairs, {subset["Trades"].sum()} trades, '
          f'total PnL: ${subset["PnL"].sum():,.0f}, '
          f'avg PnL/pair: ${subset["PnL"].mean():,.0f}')

print()

# Show the pairs only in 51-100 tier
print('Pairs in Top 51-100 (excluded from Top 50):')
for _, row in tier_df[tier_df['Tier']=='Top 51-100'].sort_values('PnL', ascending=False).iterrows():
    print(f'  {row["Pair"]:<22} PnL=${row["PnL"]:>8,.0f}  trades={row["Trades"]}')

# Bar chart
fig = px.bar(tier_df, x='Pair', y='PnL', color='Tier',
             title='Profit by Pair (colored by volume tier)',
             template=TEMPLATE, height=500,
             color_discrete_map={'Top 30': '#636EFA', 'Top 31-50': '#00CC96', 'Top 51-100': '#EF553B'})
fig.update_layout(xaxis_tickangle=-45, xaxis_tickfont_size=8)
fig.show()

In [ ]:
# Capital efficiency — % of equity deployed over time
def compute_daily_exposure(trades, starting_balance=STARTING_BALANCE):
    ts = trades.copy()
    ts['open_date'] = pd.to_datetime(ts['open_date'])
    ts['close_date'] = pd.to_datetime(ts['close_date'])
    ts_sorted = ts.sort_values('close_date')
    ts_sorted['equity'] = starting_balance + ts_sorted['profit_abs'].cumsum()
    date_range = pd.date_range(ts['open_date'].min().normalize(),
                               ts['close_date'].max().normalize(), freq='D')
    equity_by_close = ts_sorted.groupby(ts_sorted['close_date'].dt.normalize())['equity'].last()
    equity_series = equity_by_close.reindex(date_range, method='ffill').fillna(starting_balance)
    records = []
    for day in date_range:
        open_mask = (ts['open_date'].dt.normalize() <= day) & (ts['close_date'].dt.normalize() >= day)
        open_trades = ts[open_mask]
        n_open = len(open_trades)
        total_deployed = open_trades['stake_amount'].sum()
        equity = equity_series.loc[day]
        pct_deployed = (total_deployed / equity * 100) if equity > 0 else 0
        records.append({'date': day, 'open_trades': n_open, 'deployed_pct': pct_deployed})
    return pd.DataFrame(records)

def hex_to_rgba(hex_color, alpha=0.3):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

config_labels = list(all_trades.keys())
daily_exposure = {}
for label in config_labels:
    daily_exposure[label] = compute_daily_exposure(all_trades[label])
    de = daily_exposure[label]
    print(f'{label}: avg open={de["open_trades"].mean():.1f}, avg deployed={de["deployed_pct"].mean():.0f}%')

# Capital deployed subplot
fig = make_subplots(rows=len(config_labels), cols=1, shared_xaxes=True,
                    subplot_titles=config_labels, vertical_spacing=0.06)
for i, label in enumerate(config_labels, 1):
    de = daily_exposure[label]
    deployed_smooth = de['deployed_pct'].rolling(7, min_periods=1).mean()
    fig.add_trace(go.Scatter(
        x=de['date'], y=deployed_smooth, mode='lines',
        line=dict(color=COLORS[label], width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(COLORS[label], 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['deployed_pct'].mean()
    fig.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                  annotation_text=f'avg={mean_val:.0f}%', annotation_font_color='white',
                  row=i, col=1)
    fig.update_yaxes(title_text='Deployed %', range=[0, 120], row=i, col=1)

fig.update_layout(template=TEMPLATE, height=220 * len(config_labels),
                  title='Capital Efficiency — % of Equity Deployed (7d smoothed)')
fig.show()

## Summary

**Top 50 is the clear winner:**
- Sharpe 1.80 — best across all configs
- Calmar 0.94 — matches Arm D
- Profit $176K — close to Arm D ($212K) with tighter drawdown (-26.3% vs -29.5%)
- Zero dust trades
- All percentage-based, fully adaptive

**Key insight:** Pairs 51-100 by volume are net negative. They steal trading slots from profitable blue chips. The top 50 by historical volume is the sweet spot.